# Phase 2 - Create Bronze BTS Flight Table

The earlier notebook generated a weather-rule proxy label. That is not a valid aviation-disruption target. This replacement introduces the official BTS Reporting Carrier On-Time Performance dataset containing actual delays and cancellations.

Production counterpart: `spark_jobs/build_bronze_bts.py`.

In [1]:
import os
import sys
from pathlib import Path
import boto3
from pyspark.sql import functions as F
PROJECT_ROOT = Path("/workspace")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from spark_jobs.common import create_spark_session, load_settings, s3a_uri

YEAR, MONTH = 2024, 1
settings = load_settings()
spark = create_spark_session("phase2-inspect-bronze-bts", settings)
archive_name = f"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{YEAR}_{MONTH}.zip"
raw_archive_key = f"bts_on_time/raw_zip/{archive_name}"
destination = s3a_uri(settings.lakehouse_bucket, f"bronze/bts_on_time/year={YEAR}/month={MONTH:02d}")
print("Original BTS archive:", f"s3://{settings.raw_bucket}/{raw_archive_key}")
print("Bronze destination:", destination)

Original BTS archive: s3://raw/bts_on_time/raw_zip/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1.zip
Bronze destination: s3a://lakehouse/bronze/bts_on_time/year=2024/month=01


## Production Conversion Logic

BTS distributes one ZIP archive per month. The production job downloads only the requested month, extracts its single CSV temporarily, stages that CSV for Spark workers, casts the selected fields, writes Bronze Parquet, and deletes the temporary staged CSV.

In [2]:
from spark_jobs.build_bronze_bts import SELECTED_COLUMNS, archive_name

print("Selected official BTS source columns:")
for column in SELECTED_COLUMNS:
    print(" -", column)
print("\nArchive naming example:", archive_name(YEAR, MONTH))

Selected official BTS source columns:
 - Year
 - Month
 - FlightDate
 - Reporting_Airline
 - Flight_Number_Reporting_Airline
 - OriginAirportID
 - Origin
 - DestAirportID
 - Dest
 - CRSDepTime
 - DepDelayMinutes
 - ArrDelayMinutes
 - ArrDel15
 - Cancelled
 - Diverted
 - Distance

Archive naming example: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1.zip


To rebuild the monthly Bronze table from the original ZIP, run:

```bash
python -m spark_jobs.build_bronze_bts --year 2024 --month 1
```

The next cell inspects the verified output without repeating ZIP extraction during every presentation.

In [3]:
bronze_bts_df = spark.read.parquet(destination)
bronze_bts_df.printSchema()
bronze_bts_df.show(10, truncate=False)
print("Stored January BTS flight rows:", bronze_bts_df.count())

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- flight_date: date (nullable = true)
 |-- reporting_airline: string (nullable = true)
 |-- flight_number: string (nullable = true)
 |-- origin_airport_id: integer (nullable = true)
 |-- origin: string (nullable = true)
 |-- destination_airport_id: integer (nullable = true)
 |-- destination: string (nullable = true)
 |-- scheduled_departure_hhmm: string (nullable = true)
 |-- scheduled_departure_hour_local: integer (nullable = true)
 |-- departure_delay_minutes: double (nullable = true)
 |-- arrival_delay_minutes: double (nullable = true)
 |-- arrival_delayed_15_minutes: double (nullable = true)
 |-- cancelled: double (nullable = true)
 |-- diverted: double (nullable = true)
 |-- distance_miles: double (nullable = true)



+----+-----+-----------+-----------------+-------------+-----------------+------+----------------------+-----------+------------------------+------------------------------+-----------------------+---------------------+--------------------------+---------+--------+--------------+
|year|month|flight_date|reporting_airline|flight_number|origin_airport_id|origin|destination_airport_id|destination|scheduled_departure_hhmm|scheduled_departure_hour_local|departure_delay_minutes|arrival_delay_minutes|arrival_delayed_15_minutes|cancelled|diverted|distance_miles|
+----+-----+-----------+-----------------+-------------+-----------------+------+----------------------+-----------+------------------------+------------------------------+-----------------------+---------------------+--------------------------+---------+--------+--------------+
|2024|1    |2024-01-03 |WN               |3151         |13495            |MSY   |12191                 |HOU        |0830                    |8                  

Stored January BTS flight rows: 547271


In [4]:
bronze_bts_df.agg(
    F.countDistinct("origin").alias("origin_airports"),
    F.countDistinct("destination").alias("destination_airports"),
    F.min("flight_date").alias("first_flight_date"),
    F.max("flight_date").alias("last_flight_date"),
    F.sum("cancelled").alias("cancelled_flights"),
).show(truncate=False)

+---------------+--------------------+-----------------+----------------+-----------------+
|origin_airports|destination_airports|first_flight_date|last_flight_date|cancelled_flights|
+---------------+--------------------+-----------------+----------------+-----------------+
|334            |334                 |2024-01-01       |2024-01-31      |20389.0          |
+---------------+--------------------+-----------------+----------------+-----------------+



## Verified BTS Bronze Result

The January 2024 Bronze BTS partition contains **547,271 real flight rows**.

In [5]:
spark.stop()